# 04 — Operations & Broadcasting
**Goal:** Do math on arrays without writing a single loop.

```
Vectorized ops   → operator applies element-wise
Broadcasting     → arrays of different shapes work together automatically
ufuncs           → universal functions (np.sqrt, np.log, np.exp, ...)
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

## 1. Element-wise arithmetic

In [ ]:
sessions     = np.array([3200, 2800, 3500, 2600, 4100])
activations  = np.array([  96,   59,  158,   47,  156])

# These all work element-by-element — no loops
cvr          = activations / sessions * 100   # conversion rate %
incremental  = sessions - sessions.mean()     # deviation from mean
double       = sessions * 2
sessions_k   = sessions / 1000               # sessions in thousands

print('CVR %:      ', np.round(cvr, 2))
print('Δ from mean:', np.round(incremental, 0))
print('sessions K: ', sessions_k)

In [ ]:
# Comparison operators return boolean arrays
above_avg = sessions > sessions.mean()
print('Above avg:', above_avg)

# Equality
channels = np.array(['organic', 'paid', 'email', 'social', 'direct'])
print('Is email:', channels == 'email')

## 2. Broadcasting — the core rule

In [ ]:
# Scalar broadcasts over the whole array — you already know this:
arr = np.array([1, 2, 3, 4, 5])
print(arr * 10)   # [10, 20, 30, 40, 50]

# More interesting: 2D × 1D
# funnel: (3 metrics) × (5 channels)
funnel = np.array([
    [3200, 2800, 3500, 2600, 4100],
    [2400, 1960, 2800, 1820, 3280],
    [  96,   59,  158,   47,  156],
])

# Divide every row by the sessions row to get step-by-step rates
# sessions row shape: (5,) → broadcasts to (3, 5)
rates = funnel / funnel[0]   # each row ÷ sessions
print(np.round(rates, 3))

In [ ]:
# Broadcasting rule: shapes are compatible if, reading right-to-left,
# each pair of dims is either equal or one of them is 1.

# Column normalization: divide each column by its max
col_max = funnel.max(axis=0)   # shape (5,) — max per channel
print('col_max shape:', col_max.shape)
normalized = funnel / col_max
print(np.round(normalized, 3))

In [ ]:
# Row normalization: divide each row by its max
# Need shape (3, 1) to broadcast across columns
row_max = funnel.max(axis=1, keepdims=True)  # shape (3, 1)
print('row_max shape:', row_max.shape)
normalized_rows = funnel / row_max
print(np.round(normalized_rows, 3))

## 3. Universal functions (ufuncs)

In [ ]:
cvr = np.array([3.0, 2.1, 4.5, 1.8, 3.8])

# Math ufuncs — all vectorized
print('sqrt:    ', np.sqrt(cvr))
print('log:     ', np.round(np.log(cvr), 3))     # natural log
print('log10:   ', np.round(np.log10(cvr), 3))
print('exp:     ', np.round(np.exp(cvr), 2))
print('abs:     ', np.abs(cvr - cvr.mean()))      # absolute deviation
print('clip:    ', np.clip(cvr, 2.0, 4.0))        # cap between 2 and 4

In [ ]:
# Rounding
print(np.round(cvr, 1))
print(np.floor(cvr))   # round down
print(np.ceil(cvr))    # round up

In [ ]:
# Trigonometry (useful for signal processing / seasonality)
t = np.linspace(0, 2 * np.pi, 12)   # one year, monthly
seasonal = np.sin(t)
print(np.round(seasonal, 2))

## 4. Practical example — channel performance index

In [ ]:
df = pd.read_csv('data/raw/funnel_data.csv')

sessions    = df['visita_landing'].to_numpy(dtype=float)
activations = df['activacion_tarjeta'].to_numpy(dtype=float)
channel     = df['channel'].to_numpy()

# End-to-end CVR
cvr = activations / sessions * 100

# Z-score normalize CVR: (x - mean) / std
cvr_z = (cvr - cvr.mean()) / cvr.std()

# Volume score: log(sessions) normalized
vol = np.log(sessions)
vol_z = (vol - vol.mean()) / vol.std()

# Simple composite performance index
perf = 0.6 * cvr_z + 0.4 * vol_z

# Top 5 records
top5_idx = np.argsort(perf)[::-1][:5]
print('Top 5 rows by perf index:')
for i in top5_idx:
    print(f'  row {i:3d} | channel={channel[i]:8s} | CVR={cvr[i]:.2f}% | score={perf[i]:.3f}')

## Summary
| Concept | Example |
|---|---|
| Element-wise ops | `a + b`, `a * 2`, `a / b` |
| Comparisons | `a > 0`, `a == b` → bool array |
| Broadcasting | `(3,5) / (5,)` → works row-wise |
| keepdims | `arr.max(axis=1, keepdims=True)` → `(n,1)` |
| ufuncs | `np.sqrt`, `np.log`, `np.exp`, `np.clip` |
| Z-score | `(x - x.mean()) / x.std()` |

**Next:** `05_aggregations.ipynb` — sum, mean, std, and axis-based reductions.